# Part 7 - Optuna Hyperparameter Tuning
Tune CatBoost, LightGBM, and XGBoost using Optuna.

In [ ]:
# !pip install optuna -q

import optuna
import numpy as np

from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score

TARGET='addicted_label'
ID='id'

FEATURES=[c for c in train_fe.columns if c not in [ID,TARGET]]
X=train_fe[FEATURES]
y=train_fe[TARGET]

cv=StratifiedKFold(n_splits=3,shuffle=True,random_state=42)


## CatBoost Objective

In [ ]:
from catboost import CatBoostClassifier

cat_cols=X.select_dtypes(include=['object','category']).columns.tolist()
cat_idx=[X.columns.get_loc(c) for c in cat_cols]

def objective_cat(trial):

    params={
        'loss_function':'Logloss',
        'eval_metric':'AUC',
        'iterations':2000,
        'learning_rate':trial.suggest_float('learning_rate',0.01,0.2,log=True),
        'depth':trial.suggest_int('depth',4,10),
        'l2_leaf_reg':trial.suggest_float('l2_leaf_reg',1,20),
        'subsample':trial.suggest_float('subsample',0.6,1.0),
        'verbose':False,
        'random_seed':42
    }

    scores=[]

    for tr,va in cv.split(X,y):

        model=CatBoostClassifier(**params)

        model.fit(
            X.iloc[tr],y.iloc[tr],
            cat_features=cat_idx,
            eval_set=(X.iloc[va],y.iloc[va]),
            early_stopping_rounds=200,
            use_best_model=True
        )

        pred=model.predict_proba(X.iloc[va])[:,1]
        scores.append(roc_auc_score(y.iloc[va],pred))

    return np.mean(scores)

study_cat=optuna.create_study(direction='maximize')
study_cat.optimize(objective_cat,n_trials=30)

print(study_cat.best_value)
print(study_cat.best_params)


## LightGBM Objective

In [ ]:
from lightgbm import LGBMClassifier

X_lgb=X.copy()
cat_cols=X_lgb.select_dtypes(include=['object','category']).columns.tolist()
for c in cat_cols:
    X_lgb[c]=X_lgb[c].astype('category')

def objective_lgb(trial):

    params={
        'objective':'binary',
        'metric':'auc',
        'n_estimators':3000,
        'learning_rate':trial.suggest_float('learning_rate',0.01,0.15,log=True),
        'num_leaves':trial.suggest_int('num_leaves',31,255),
        'min_child_samples':trial.suggest_int('min_child_samples',20,150),
        'subsample':trial.suggest_float('subsample',0.6,1.0),
        'colsample_bytree':trial.suggest_float('colsample_bytree',0.6,1.0),
        'random_state':42
    }

    scores=[]

    for tr,va in cv.split(X_lgb,y):
        model=LGBMClassifier(**params)
        model.fit(X_lgb.iloc[tr],y.iloc[tr],categorical_feature=cat_cols)
        pred=model.predict_proba(X_lgb.iloc[va])[:,1]
        scores.append(roc_auc_score(y.iloc[va],pred))

    return np.mean(scores)

study_lgb=optuna.create_study(direction='maximize')
study_lgb.optimize(objective_lgb,n_trials=30)

print(study_lgb.best_value)
print(study_lgb.best_params)


## XGBoost Objective

In [ ]:
from xgboost import XGBClassifier
from sklearn.preprocessing import OrdinalEncoder

X_xgb=X.copy()
cat_cols=X_xgb.select_dtypes(include=['object','category']).columns.tolist()

enc=OrdinalEncoder(handle_unknown='use_encoded_value',unknown_value=-1)

if len(cat_cols):
    X_xgb[cat_cols]=enc.fit_transform(X_xgb[cat_cols].astype(str))

def objective_xgb(trial):

    params={
        'objective':'binary:logistic',
        'eval_metric':'auc',
        'tree_method':'hist',
        'n_estimators':3000,
        'learning_rate':trial.suggest_float('learning_rate',0.01,0.15,log=True),
        'max_depth':trial.suggest_int('max_depth',4,10),
        'min_child_weight':trial.suggest_int('min_child_weight',1,10),
        'subsample':trial.suggest_float('subsample',0.6,1.0),
        'colsample_bytree':trial.suggest_float('colsample_bytree',0.6,1.0),
        'gamma':trial.suggest_float('gamma',0,5),
        'random_state':42,
        'n_jobs':-1
    }

    scores=[]

    for tr,va in cv.split(X_xgb,y):
        model=XGBClassifier(**params)
        model.fit(X_xgb.iloc[tr],y.iloc[tr],verbose=False)
        pred=model.predict_proba(X_xgb.iloc[va])[:,1]
        scores.append(roc_auc_score(y.iloc[va],pred))

    return np.mean(scores)

study_xgb=optuna.create_study(direction='maximize')
study_xgb.optimize(objective_xgb,n_trials=30)

print(study_xgb.best_value)
print(study_xgb.best_params)


## Save Best Parameters

In [ ]:
best_params={
    'catboost':study_cat.best_params,
    'lightgbm':study_lgb.best_params,
    'xgboost':study_xgb.best_params
}

import json

with open('best_params.json','w') as f:
    json.dump(best_params,f,indent=4)

print(best_params)
print('Saved best_params.json')


### Next
Part 8 will train all three models using the tuned parameters, generate OOF predictions, and create a weighted ensemble.